## *A neural network potential for the Zundel ion*


**In a few words -** In this tutorial, we will learn how to train and use an interatomic potential based on a neural network, using a DFTB molecular dynamics trajectory of the zundel ion (H<sub>5</sub>O<sub>2</sub><sup>+</sup>) in vacuum as a dataset.  

**In greater detail -** Here, you will be able to cover all the important steps of constructing a machine-learned interatomic potential, from selecting an appropriate model to using it in a well-established MD software. We will focus on an active learning procedure to optimize the quality of a training set, using the "[query by committee](https://aip.scitation.org/doi/10.1063/5.0016004)" approach. Instead of training a single neural network, we'll train an ensemble of models (typically 5-10), called a committee. The interest is twofold:
* When running new dynamics, the standard deviation between the various models (called the committee disagreement") provides an estimate of the uncertainty of the model prediction for a given configuration.
* When training, the query by committee approach will allow us to select an optimal training set. This is an iterative process, which works in the following way: first, a small $\mathcal{O}(10^1)$, random ensemble of configurations is selected as training set out of a larger $\mathcal{O}(10^4)$ ensemble of configurations, typically a long trajectory. The committee is trained. The model is then tested on a medium $\mathcal{O}(10^2)$ ensemble of configurations. Based on the largest committee disagreement, new structures ($\mathcal{O}(10^1)$) are added to the training set. This is repeated until convergence in the committee disagreement is achieved.

Committee member models are trained on slightly different subsets of the same training set. Together with the stochasticity of weights initialization and of the optimization algorithm, this is enough to generate a sufficiently diverse committee of models.

**Tools used -** [aml](https://github.com/MarsalekGroup/aml) active learning of machine-learned potentials, based on [n2p2](https://compphysvienna.github.io/n2p2/), a package for neural network potentials. [cp2k](https://www.cp2k.org/), a general molecular dynamics package.

**How to use this?** This tutorial uses Python, some familiarity with programming would be beneficial but is not required. Run each code cell in order by clicking "Shift + Enter". Feel free to modify the code, or change queries to familiarize yourself with the workings on the code.

Suggested modifications and exercises are included in <font color=blue> blue</font>.

**Outline:**

1. Loading and visualizing the dataset
2. Training a one-member committee model, and evaluating it
2. Selecting an appropriate training set using active learning
3. Training the final model
4. Running molecular dynamics
5. Assessing the quality of the model

**Get started:** Click "Shift-Enter" on the code cells to run!

**Setup (Google Colab).** Run the next cell first, and again whenever the Colab runtime restarts. It downloads the course files (or uses `TP-MLIP.zip` if you uploaded it in the Files panel on the left), installs n2p2, CP2K and the Python packages we need (1 to 2 minutes the first time), and moves into this tutorial's folder.

In [ ]:
ZIP_URL = 'https://github.com/zacway/TP-MLIP/releases/latest/download/TP-MLIP.zip'

import os
if not os.path.exists('/content/TP-MLIP'):
    # a TP-MLIP.zip uploaded by hand (Files panel on the left) is used instead of downloading it
    if not os.path.exists('/content/TP-MLIP.zip'):
        !curl -Lsf -o /content/TP-MLIP.zip.part "$ZIP_URL" && mv /content/TP-MLIP.zip.part /content/TP-MLIP.zip
    assert os.path.exists('/content/TP-MLIP.zip'), 'Could not download TP-MLIP.zip: upload it with the Files panel on the left, then run this cell again.'
    !unzip -q /content/TP-MLIP.zip -d /content
%run /content/TP-MLIP/colab_setup.py
%cd /content/TP-MLIP/tutorial_01_zundel

In [ ]:
# We'll start by loading and visualizing the reference DFTB trajectory

%matplotlib inline

import os

base_dir = os.getcwd()

# Some paths and files for the reference DFTB data

ref_dir = base_dir+'/data/'

ref_trj = ref_dir+'zundel-pos-1.xyz'
ref_frc = ref_dir+'zundel-frc-1.xyz'
ref_vel = ref_dir+'zundel-vel-1.xyz'
ref_ene = ref_dir+'zundel-1.ener'
ref_top = ref_dir+'zundel.pdb'

# Some paths we'll use later on

sin_dir = base_dir+'/single/'
qbc_dir = base_dir+'/qbc/'
mod_dir = base_dir+'/train/'

# Some conversion constants

au_to_evang = 51.42208619083232
au_to_ev = 27.21138386

# Let's visualize the trajectory.
# Both linear and angular momenta of the system are removed.

import mdtraj as mdt

show_md(mdt.load_xyz(ref_trj, top=ref_top, stride=10))

In [ ]:
# Now let's look at some thermodynamical quantities over the course of the trajectory

os.chdir(base_dir)

import numpy as np

def compute_time(thermo, n_cpus):
    t_min = np.sum(thermo[6,:])/60
    n_steps = int(thermo[-1,0])
    rate = int(n_steps/t_min/n_cpus)
    print('The full simulation took {:02.2f} minutes using {} CPU, for a total of {} MD steps'\
          ', corresponding to an average compute rate of {} MD steps per minute per CPU. '\
          .format(t_min, n_cpus, n_steps, rate))

import matplotlib.pyplot as plt

def plot_thermo(thermo):
    fig, axs = plt.subplots(2, 1)

    for ax in axs.flat:
        ax.set_xlim([0.0, 1e-3*thermo[-1,1]])

    axs[0].set_ylabel(r'$E_{p}$ (eV)')
    axs[0].plot(1e-3*thermo[::100,1], au_to_ev*thermo[::100,4], color='firebrick')

    axs[1].set_xlabel(r't (ps)')
    axs[1].set_ylabel(r'T (K)')
    axs[1].plot(1e-3*thermo[::100,1], thermo[::100,3], color='dodgerblue')
    axs[1].hlines(np.average(thermo[:,3]), 1e-3*thermo[0,1], 1e-3*thermo[-1,1], alpha=0.5, color='gray')

ref_thermo = np.loadtxt(ref_ene)

compute_time(ref_thermo, 1)

plot_thermo(ref_thermo)

**A naive approach:** Here we'll train a single model, using a training set of 500 data points evenly sampled from the trajectory.

In [ ]:
os.chdir(base_dir)

import aml

def read_structures(trj_file, frc_file, stride):
    # stride through trajectory

    print('Reading structures')
    print('------------------')
    print()
    print(f'Stride: {stride}')
    print()

    frames = aml.read_frames_cp2k(fn_positions=trj_file, fn_forces=frc_file)
    structures = aml.Structures.from_frames(frames, stride=stride, probability=1.0)

    print(f'{len(structures):} structures kept')
    print()

    return structures

import sys
import fileinput

def set_nn(nn_file, n_hidden, n_neurons, activ):
    for line in fileinput.input(nn_file, inplace=True):
        if line.strip().startswith('global_hidden_layers_short'):
            line = 'global_hidden_layers_short      '+str(n_hidden)+' \n'
        if line.strip().startswith('global_nodes_short'):
            line = 'global_nodes_short              '+str(n_neurons)+' \n'
        if line.strip().startswith('global_activation_short'):
            line = 'global_activation_short         '+str(activ)+' \n'
        sys.stdout.write(line)

# Prepare the neural network architecture

n_hidden = 2
n_neurons = '5 5 '
activ = 't t l'

n_members = 1

set_nn(sin_dir+'input.nn', n_hidden, n_neurons, activ)

# Settings - n2p2 constructor
kwargs_model = dict(
    elements = ['O', 'H'],
    n = n_members,
    fn_template = sin_dir+'input.nn',
    exclude_triples = [['O', 'O', 'O']],
    n_tasks = 1,
    n_core_task = 1,
)

# Our training set
structures = read_structures(ref_trj, ref_frc, 25)

os.chdir(sin_dir)
# Initialize the model, perform training, save the final model.
print('Creating blank NNP model... ', end='', flush=True)

n2p2 = aml.N2P2(dir_run=sin_dir+'training', **kwargs_model)
print('done.')
print()

print('Training all member models... ', end='', flush=True)
n2p2.train(structures, n_epoch=50)
print('done.')
print()

print('Saving all member models... ', end='', flush=True)
n2p2.save_model()
print('done.')
print()
os.chdir(base_dir)

In [ ]:
# Plot learning curves:

def plot_optimization(opt_dir, n_members):
    fig, axs = plt.subplots(2, 1, figsize=(8,8))

    c_list = iter(plt.cm.rainbow(np.linspace(0,1,n_members)))

    for co in range(n_members):
        lea_data = np.loadtxt(opt_dir+'/train-'+f'{co:03d}'+'/learning-curve.out', usecols=(0,11,12))
        col = next(c_list)
        axs[0].plot(lea_data[:,0], au_to_evang*lea_data[:,1], alpha=0.8, color=col, label='Member '+str(co))
        axs[1].plot(lea_data[:,0], au_to_evang*lea_data[:,2], alpha=0.8, color=col)

    for ax in axs.flat:
        ax.set_xlim([lea_data[0,0], lea_data[-1,0]])
        ax.set_yscale('log')

    axs[1].set_xlabel(r'Epoch')
    axs[0].set_ylabel(r'MAE train forces (eV/$\AA$)')
    axs[1].set_ylabel(r'MAE test forces (eV/$\AA$)')
    axs[0].legend()

plot_optimization(sin_dir+'training', n_members)

**We can now run some dynamics based on our new model.**

Run the next cell to start a CP2K MD simulation in the "single" subfolder. It prints the latest line of the energy file every 15 s (step, time, kinetic energy, temperature, potential energy). Once it reports "Done", you can proceed to the next cell.

In [ ]:
run_cp2k('zundel.inp', sin_dir)

In [ ]:
# We'll visualize the new trajectory data now

sin_thermo = np.loadtxt(sin_dir+'zundel-1.ener')
plot_thermo(sin_thermo)

show_md(mdt.load_xyz(sin_dir+'zundel-pos-1.xyz', top=ref_top, stride=10))

* <font color=blue>What is happening? Come up with a few hypotheses which could explain the phenomenon observed. </font>

In [ ]:
# Next we'll perform active learning, and use multiple models: an optimal training set
# will be constructed using the "Query by Committee" algorithm.

os.chdir(base_dir)

# Prepare the neural network architecture
# Edit the three following lines if you want to experiment with different layouts

n_hidden = 2
n_neurons = '5 5 '
activ = 't t l'

n_iterations = 15
n_members = 8

set_nn(qbc_dir+'input.nn', n_hidden, n_neurons, activ)

# Settings - n2p2 constructor
kwargs_model = dict(
    elements = ['O', 'H'],
    n = n_members,
    fn_template = qbc_dir+'input.nn',
    exclude_triples = [['O', 'O', 'O']],
    n_tasks = 8,
    n_core_task = 1,
    remove_output = True
)

# Structures to select from
structures = read_structures(ref_trj, ref_frc, 1)

# Parameters of the query by committee algorithm
qbc = aml.QbC(
    structures = structures,
    cls_model = aml.N2P2,
    kwargs_model = kwargs_model,
    n_train_initial = 20,
    n_add = 20,
    n_epoch = 15,
    n_iterations = n_iterations,
    n_candidate = 500,
    fn_results = 'results.shelf',
    fn_restart = None
)

os.chdir(qbc_dir)
qbc.run()
os.chdir(base_dir)


* <font color=blue>Explain differences in committor disagreement between selected, train, and candidate datasets</font>
* <font color=blue>How does the compute time increase with QbC iterations? Why? Can you say something about computational complexity?</font>

In [ ]:
# Plot the initial and final training points,
# along the potential energy trajectory.

import shelve

def plot_training(thermo, n_iterations):
    fig, ax = plt.subplots(1, 1, figsize=(14,4))

    ax.set_xlim([0.0, 1e-3*thermo[-1,1]])

    ax.set_xlabel(r't (ps)')
    ax.set_ylabel(r'$E_{p}$ (eV)')
    ax.plot(1e-3*thermo[:,1], au_to_ev*thermo[:,4], alpha=0.5, color='gray')

    with shelve.open('qbc/results.shelf') as res:
        ts_ini = res['iteration-001']
        ts_fin = res['iteration-'+f'{n_iterations:03d}']

        for ind in ts_fin['idx_train']:
            ax.scatter(1e-3*thermo[5*ind,1], au_to_ev*thermo[5*ind,4], marker='o', s=6.0, alpha=0.8, color='dodgerblue')

        for ind in ts_ini['idx_train']:
            ax.scatter(1e-3*thermo[5*ind,1], au_to_ev*thermo[5*ind,4], marker='o', s=6.0, alpha=0.8, color='firebrick')

plot_training(ref_thermo, n_iterations)

* <font color=blue>Why do selected training points seem to be distributed towards high potential energy values?</font>

In [ ]:
# Using the last training set, let's now tightly optimize our neural network model
# with a greater number of epochs. This will constitute our final model.

os.chdir(base_dir)

# Load our final training set
print('Loading training set... ', end='', flush=True)
structures = aml.Structures.from_file(qbc_dir+f'iteration-{n_iterations:03d}/train-000/input.data')
print('done.')
print()

# Initialize the model, perform training, save the final model.
os.chdir(mod_dir)
print('Creating blank NNP model... ', end='', flush=True)
# Settings - n2p2 constructor
kwargs_model = dict(
    elements = ['O', 'H'],
    n = n_members,
    fn_template = qbc_dir+'input.nn',
    exclude_triples = [['O', 'O', 'O']],
    n_tasks = 4,
    n_core_task = 1
)
n2p2 = aml.N2P2(dir_run=mod_dir+'final_training', **kwargs_model)
print('done.')
print()

print('Training all member models... ', end='', flush=True)
n2p2.train(structures, n_epoch=50)
print('done.')
print()

print('Saving all member models... ', end='', flush=True)
n2p2.save_model()
print('done.')
print()
os.chdir(base_dir)

In [ ]:
# Let's plot learning curves for each member of the final model.

plot_optimization(mod_dir+'final_training', n_members)

**Now that we have a model, let's run some MD!**

* <font color=blue>If you've modified the number of committee members, adjust the cp2k input script ("nnp_md/zundel.inp", open it from the Files panel on the left)</font>
* Run the next cell to start the CP2K MD simulation in the "nnp_md" subfolder, and wait until it reports "Done".

Something to try later on :
* <font color=blue>Change the total number of committee members (either by training a new model, or by directly removing members in the cp2k input script. How does the compute time change? Why?</font>

In [ ]:
run_cp2k('zundel.inp', base_dir+'/nnp_md')

In [ ]:
# Time to visualize our newly-generated trajectory.

os.chdir(base_dir)

nnp_dir = base_dir+'/nnp_md/'

nnp_trj = nnp_dir+'zundel-pos-1.xyz'
nnp_frc = nnp_dir+'zundel-frc-1.xyz'
nnp_vel = nnp_dir+'zundel-vel-1.xyz'
nnp_ene = nnp_dir+'zundel-1.ener'
nnp_top = nnp_dir+'zundel.pdb'

nnp_thermo = np.loadtxt(nnp_ene)

compute_time(nnp_thermo, 1)

plot_thermo(nnp_thermo)

show_md(mdt.load_xyz(nnp_trj, top=nnp_top, stride=10))

In [ ]:
# We'll now compute a few properties, to quantitatively assess
# the quality of our model.

import aml.score as mlps

dt_trj = 2.5 # Time between MD frames, in fs

# Loading position data
ref_pos = mlps.load_with_cell(ref_trj, top=ref_top)
nnp_pos = mlps.load_with_cell(nnp_trj, top=nnp_top)

# Perform RDF scoring
mlps.run_rdf_test(ref_pos, nnp_pos)

# Q's: why is the O-O error larger? Is it surprising to get such a good agreement for a
# single molecule?

In [ ]:
# Loading velocity data
ref_atv = mlps.load_with_cell(ref_vel, top=ref_top)
nnp_atv = mlps.load_with_cell(nnp_vel, top=nnp_top)

# Perform VDOS scoring
mlps.run_vdos_test(ref_atv, dt_trj, nnp_atv, dt_trj)

<font color=blue>Compare the number of structures in the training sets of the one-member model and of the eight-members model. You can look in the output of the QbC run in the notebook, or dig in the output files under "train/final_training" and "single/training". For a given sampling of phase space, for instance through molecular dynamics, are all configurations equally valuable to train a neural network model? </font>

**Is our model transferable?** We'll run an MD simulation on a larger cluster of protonated water ( 6H<sub>2</sub>O+H<sup>+</sup>) to check.

* <font color=blue>If you've modified the number of committee members, adjust the cp2k input script ("cluster/cluster.inp")</font>
* Run the next cell to start the CP2K MD simulation in the "cluster" subfolder, and wait until it reports "Done".

In [ ]:
run_cp2k('cluster.inp', base_dir+'/cluster')

In [ ]:
# Visualize the dynamics first.
# This time, the run is only 500 fs long (1000 iterations).

clu_dir = base_dir+'/cluster'

os.chdir(clu_dir)

# Plot thermo quantities too
clu_ene = np.loadtxt(clu_dir+'/cluster-1.ener')
plot_thermo(clu_ene)

show_md(mdt.load_xyz(\
            clu_dir+'/cluster-pos-1.xyz',\
            top=clu_dir+'/cluster.pdb', stride=1))


<font color=blue>What is happening? Can you think of reasons why?</font> To see if we can make things better, move on to the second tutorial.